# Lane Mark Focus — 5-fold CV (최종 베스트 모델)

Fold 0 성공 (Lane IoU 0.36, mIoU 0.79) → 동일 레시피로 5-fold 재학습.

**설정** (fold 0 과 동일):
- Morphology dilation 5px (Lane Mark GT 두께화)
- 0.3 Focal(γ=2) + 0.3 Dice + 0.4 WeightedCE (class_weight[3]=50)
- 768×768, BS=4, 40 epochs
- Seed 42

예상 시간: ~**72분** (fold당 ~14~15분 × 5).


In [ ]:
import sys, torch, numpy as np, random, time, json, cv2
from pathlib import Path
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

BASE = Path('.').resolve()   # 이 노트북이 있는 train/ 폴더에서 실행
import dataset as ds_mod

device = 'cuda'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
class DilatedLaneDataset(Dataset):
    def __init__(self, image_ids, transform=None, lane_dilate=5):
        splits = ds_mod.load_splits()
        id_to_file = splits['image_id_to_file_name']
        self.items = [{'image_id': i, 'file_name': id_to_file[str(i)]} for i in image_ids]
        self.transform = transform
        self.lane_dilate = lane_dilate
        self.kernel = np.ones((lane_dilate, lane_dilate), np.uint8) if lane_dilate > 0 else None

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        item = self.items[idx]
        img = np.array(Image.open(ds_mod.IMG_DIR / item['file_name']).convert('RGB'))
        mask = np.array(Image.open(ds_mod.MASKS_DIR / item['file_name']))
        if self.kernel is not None:
            lane = (mask == 3).astype(np.uint8)
            dilated = cv2.dilate(lane, self.kernel, iterations=1)
            expand_into = ((mask == 0) | (mask == 2))
            mask[(dilated == 1) & expand_into] = 3
        if self.transform is not None:
            out = self.transform(image=img, mask=mask)
            img, mask = out['image'], out['mask']
        if isinstance(mask, torch.Tensor):
            mask = mask.long()
        return img, mask


IMG_SIZE = 768
BATCH_SIZE = 4
NUM_CLASSES = 7
EPOCHS = 40
LR = 1e-4; WD = 1e-4
N_FOLDS = 5

def build_transforms(aug=True, size=IMG_SIZE):
    base = [A.Resize(size, size)]
    if aug:
        base += [
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
            A.CLAHE(clip_limit=3.0, p=0.3),
            A.GaussNoise(var_limit=(5.0, 25.0), p=0.15),
        ]
    base += [A.Normalize(mean=ds_mod.IMAGENET_MEAN, std=ds_mod.IMAGENET_STD), ToTensorV2()]
    return A.Compose(base)

In [ ]:
import segmentation_models_pytorch as smp

class_weights = torch.tensor([0.0, 0.5, 0.5, 50.0, 2.0, 0.5, 1.0], dtype=torch.float32).to(device)

focal_fn = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
dice_fn = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
wce_fn = nn.CrossEntropyLoss(weight=class_weights)

def combined_loss(logits, targets):
    return 0.3*focal_fn(logits, targets) + 0.3*dice_fn(logits, targets) + 0.4*wce_fn(logits, targets)

def build_model():
    return smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES).to(device)

class IoUMeter:
    def __init__(self, n): self.n = n; self.reset()
    def reset(self):
        self.inter = np.zeros(self.n, dtype=np.int64); self.union = np.zeros(self.n, dtype=np.int64)
    def update(self, p, t):
        p = p.detach().cpu().numpy().ravel(); t = t.detach().cpu().numpy().ravel()
        for c in range(self.n):
            pc = (p == c); tc = (t == c)
            self.inter[c] += np.logical_and(pc, tc).sum()
            self.union[c] += np.logical_or(pc, tc).sum()
    def compute(self):
        with np.errstate(divide='ignore', invalid='ignore'):
            return np.where(self.union > 0, self.inter/np.maximum(self.union, 1), np.nan)


def train_one_fold(fold_idx):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    splits = ds_mod.load_splits()
    fd = splits['folds'][fold_idx]
    train_ds = DilatedLaneDataset(fd['train_image_ids'], transform=build_transforms(aug=True), lane_dilate=5)
    val_ds   = DilatedLaneDataset(fd['val_image_ids'],   transform=build_transforms(aug=False), lane_dilate=0)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = {'train_loss':[], 'val_loss':[], 'val_miou':[], 'val_per_class_iou':[]}
    best_miou = -1.0; best_lane = -1.0
    best_path = BASE / f'lane_focus_fold{fold_idx}_best.pth'
    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        model.train()
        tr = []
        for imgs, msks in train_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            logits = model(imgs)
            loss = combined_loss(logits, msks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tr.append(loss.item())
        scheduler.step()
        model.eval()
        vl = []; meter = IoUMeter(NUM_CLASSES)
        with torch.no_grad():
            for imgs, msks in val_loader:
                imgs, msks = imgs.to(device), msks.to(device)
                logits = model(imgs)
                vl.append(combined_loss(logits, msks).item())
                meter.update(logits.argmax(1), msks)
        ciou = meter.compute()
        miou_fg = float(np.nanmean(ciou[1:]))
        lane = float(ciou[3]) if not np.isnan(ciou[3]) else 0.0
        history['train_loss'].append(float(np.mean(tr)))
        history['val_loss'].append(float(np.mean(vl)))
        history['val_miou'].append(miou_fg)
        history['val_per_class_iou'].append([None if np.isnan(v) else float(v) for v in ciou])
        if miou_fg > best_miou:
            best_miou = miou_fg
            torch.save({'state_dict': model.state_dict(), 'epoch': ep, 'miou': miou_fg, 'lane_iou': lane, 'fold': fold_idx}, best_path)
        if lane > best_lane: best_lane = lane
    t_total = time.time() - t0
    print(f'[Fold {fold_idx}] best mIoU={best_miou:.4f}, Lane={best_lane:.4f}, {t_total/60:.1f}min')
    return {'fold':fold_idx, 'best_miou_fg':best_miou, 'best_lane_iou':best_lane, 'history':history, 'time_min':t_total/60}

## 5-fold 순차 학습

In [ ]:
all_folds = []
t_all = time.time()
for f in range(N_FOLDS):
    r = train_one_fold(f)
    all_folds.append(r)
    with open(BASE / 'lane_focus_5fold_results.json', 'w', encoding='utf-8') as fp:
        json.dump(all_folds, fp, indent=2, ensure_ascii=False)
print(f'\n=== 전체 시간: {(time.time()-t_all)/60:.1f}분 ===')

best_mious = [r['best_miou_fg'] for r in all_folds]
best_lanes = [r['best_lane_iou'] for r in all_folds]
print(f'\nmIoU(fg): 평균={np.mean(best_mious):.4f} ± {np.std(best_mious):.4f}')
print(f'Lane IoU: 평균={np.mean(best_lanes):.4f} ± {np.std(best_lanes):.4f}')
for r in all_folds:
    print(f'  fold {r["fold"]}: mIoU={r["best_miou_fg"]:.4f}, Lane={r["best_lane_iou"]:.4f}')

In [ ]:
# ==== Lv1 5-fold 대비 ====
class_map = ds_mod.load_class_map()
label_to_name = {int(k): v for k, v in class_map['label_to_name'].items()}

with open(BASE / 'lv1_kfold_results.json') as f:
    lv1 = json.load(f)

print(f'{"Class":<14} {"Lv1 평균":>14} {"LaneFocus 평균":>18} {"Δ":>10}')
print('-' * 60)
for c in range(NUM_CLASSES):
    lv1_vals = []
    new_vals = []
    for i, r in enumerate(lv1):
        last = r['history']['val_per_class_iou'][-1]
        lv1_vals.append(last[c] if last[c] is not None else 0)
    for r in all_folds:
        last = r['history']['val_per_class_iou'][-1]
        new_vals.append(last[c] if last[c] is not None else 0)
    m_lv1 = np.mean(lv1_vals); m_new = np.mean(new_vals)
    d = m_new - m_lv1
    tag = ' 🔺' if d > 0.02 else (' 🔻' if d < -0.02 else '')
    print(f'{label_to_name[c]:<14} {m_lv1:>14.4f} {m_new:>18.4f} {d:+.4f}{tag}')

print(f'{"mIoU(fg)":<14} {np.mean([r["best_miou_fg"] for r in lv1]):>14.4f} {np.mean(best_mious):>18.4f} {np.mean(best_mious) - np.mean([r["best_miou_fg"] for r in lv1]):+.4f}')

## 시각화

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mIoU curve per fold
for r in all_folds:
    axes[0].plot(range(1, EPOCHS+1), r['history']['val_miou'], alpha=0.55, label=f'fold {r["fold"]}')
axes[0].plot(range(1, EPOCHS+1), np.mean([r['history']['val_miou'] for r in all_folds], axis=0),
             'k-', linewidth=2.5, label='평균')
# Lv1 평균도 overlay
lv1_miou_curves = np.array([r['history']['val_miou'] for r in lv1])
axes[0].plot(range(1, lv1_miou_curves.shape[1]+1), lv1_miou_curves.mean(axis=0), '--', color='tab:red', alpha=0.7, label='Lv1 평균')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val mIoU(fg)')
axes[0].set_title('Lane Focus 5-fold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Lane IoU per-fold per-epoch
for r in all_folds:
    lanes = [v[3] if v[3] is not None else 0 for v in r['history']['val_per_class_iou']]
    axes[1].plot(range(1, EPOCHS+1), lanes, alpha=0.55, label=f'fold {r["fold"]}')
mean_lane = np.mean([[v[3] if v[3] is not None else 0 for v in r['history']['val_per_class_iou']] for r in all_folds], axis=0)
axes[1].plot(range(1, EPOCHS+1), mean_lane, 'k-', linewidth=2.5, label='평균')
axes[1].axhline(0.0, color='tab:red', ls='--', alpha=0.5, label='Lv1 Lane 평균 (~0)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Lane Mark IoU')
axes[1].set_title('Lane Mark IoU 추이')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(BASE / 'lane_focus_5fold_curves.png', dpi=110, bbox_inches='tight')
plt.show()
print('저장: lane_focus_5fold_curves.png')